# importing libraries

In [1]:
!pip install gensim numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 27.4 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from gensim.models import KeyedVectors
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, f1_score
import gensim.downloader as api

# loading dataset

In [3]:
df = pd.read_csv("/content/spam.csv", encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# preprocessing

In [4]:
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)
df.rename(columns={"v1": "label", "v2": "text"}, inplace=True)
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

,label,text,label_enc
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [5]:
X_train, X_test, y_train, y_test = train = train_test_split(df['text'], df['label_enc'], test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = X_train.to_numpy(), X_test.to_numpy(), y_train.to_numpy(), y_test.to_numpy()

In [6]:
y_train = to_categorical(y_train, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)

# vocabulary stats

In [7]:
averageWordsLength = round(sum([len(i.split()) for i in df['text']]) / len(df['text']))
totalWordsLength = len(set(" ".join(df['text']).split()))
print(f"Data Loaded. Training samples: {len(X_train)}")
print(f"Average words per message: {averageWordsLength}")
print(f"Approximate vocabulary size: {totalWordsLength}")

Data Loaded. Training samples: 4457
Average words per message: 16
Approximate vocabulary size: 15686


# more preprocessing

In [8]:
text_vec = tf.keras.layers.TextVectorization(
    max_tokens=totalWordsLength,
    standardize="lower_and_strip_punctuation",
    output_mode="int",
    output_sequence_length=averageWordsLength,
)
text_vec.adapt(X_train)

X_train = text_vec(X_train)
X_test = text_vec(X_test)

In [9]:
vocab_size = text_vec.get_vocabulary()
word_index = dict(zip(vocab_size, range(len(vocab_size))))

In [10]:
ft = api.load("fasttext-wiki-news-subwords-300")

[==================================================] 100.0% 958.5/958.4MB downloaded


In [11]:
embedding_dim = 300
embedding_matrix = np.zeros((len(vocab_size), embedding_dim))

for word, i in word_index.items():
    if word in ft:
        embedding_matrix[i] = ft[word]

embedding_layer = tf.keras.layers.Embedding(
    input_dim=len(vocab_size),
    output_dim=embedding_dim,
    input_length=averageWordsLength,
    weights=[embedding_matrix],
    trainable=False
)

X_train = embedding_layer(X_train)
X_test = embedding_layer(X_test)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [12]:
y_train

array([[0., 1.],
       [1., 0.],
       [1., 0.],
       ...,
       [1., 0.],
       [1., 0.],
       [1., 0.]])

In [24]:
X_train

array([[[-3.0022e-02,  1.0605e-01,  7.5141e-02, ...,  2.0334e-02,
          1.7208e-02, -1.4516e-01],
        [-1.2973e-02, -2.9289e-01,  4.9091e-02, ..., -8.4461e-02,
         -1.4082e-02, -6.7256e-02],
        [ 1.5336e-02, -5.7898e-02,  1.9616e-02, ...,  7.9795e-02,
          1.6029e-02,  4.7662e-02],
        ...,
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00, ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00, ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00, ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00]],

       [[-6.0418e-02,  6.9955e-02,  5.3173e-02, ..., -4.7788e-03,
          3.3832e-02, -2.0630e-01],
        [ 4.5557e-03,  4.1912e-02,  6.9680e-02, ...,  7.0573e-02,
          4.4381e-02,  2.5133e-02],
        [ 1.0108e-02,  2.0139e-01, -2.8939e-02, ...,  3.2643e-02,
         -2.6049e-02, -1.4416e-02],
        ...,
        [-3.5752e-02,  8.0531e-02,  3.8187e-03, ...,  

In [14]:
train_counts = y_train.sum(axis=0)
test_counts = y_test.sum(axis=0)

print(f"Training set - Ham: {int(train_counts[0])}, Spam: {int(train_counts[1])}")
print(f"Testing set - Ham: {int(test_counts[0])}, Spam: {int(test_counts[1])}")

num_classes = y_train.shape[1]
print(f"\nTotal number of unique classes: {num_classes}")

Training set - Ham: 3859, Spam: 598
Testing set - Ham: 966, Spam: 149

Total number of unique classes: 2


In [15]:
train_labels = np.argmax(y_train, axis=1)

X_train_0 = X_train.numpy()[train_labels == 0]
X_train_1 = X_train.numpy()[train_labels == 1]

print(f"Shape of Ham training set: {X_train_0.shape}")
print(f"Shape of Spam training set: {X_train_1.shape}")

Shape of Ham training set: (3859, 16, 300)
Shape of Spam training set: (598, 16, 300)


In [16]:
majority_features = X_train_0
minority_features = X_train_1
n_majority = len(majority_features)

# oversampling minority class
indices = np.random.choice(len(minority_features), size=n_majority, replace=True)
oversampled_minority_features = minority_features[indices]

# concatenation
X_train_resampled = np.concatenate([majority_features, oversampled_minority_features])
y_train_resampled = np.concatenate([
    np.tile([1.0, 0.0], (n_majority, 1)), # labels for Ham
    np.tile([0.0, 1.0], (n_majority, 1))  # labels for Spam
])

print(f"Original training samples: {len(X_train)}")

shuffle_idx = np.random.permutation(len(X_train_resampled))
X_train = X_train_resampled[shuffle_idx]
y_train = y_train_resampled[shuffle_idx]

print(f"Resampled training samples: {len(X_train_resampled)}")
print(f"New class counts: Ham={n_majority}, Spam={n_majority}")

Original training samples: 4457
Resampled training samples: 7718
New class counts: Ham=3859, Spam=3859


# model development

In [17]:
cnn = tf.keras.models.Sequential()
cnn.add(tf.keras.layers.Input(shape=(12,300)))

In [18]:
cnn.add(tf.keras.layers.Conv1D(filters=128, kernel_size=5, activation="relu"))
cnn.add(tf.keras.layers.GlobalAveragePooling1D())
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))
cnn.add(tf.keras.layers.Dense(units=2, activation='softmax'))

In [19]:
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 8, 128)         │       192,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 208,898 (816.01 KB)

 Trainable params: 208,898 (816.01 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
cnn.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

# model training

In [21]:
history = cnn.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32)

Epoch 1/5
242/242 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.8608 - loss: 0.3142 - val_accuracy: 0.9570 - val_loss: 0.1277
Epoch 2/5
242/242 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.9700 - loss: 0.0858 - val_accuracy: 0.9677 - val_loss: 0.1142
Epoch 3/5
242/242 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.9893 - loss: 0.0363 - val_accuracy: 0.9704 - val_loss: 0.1135
Epoch 4/5
242/242 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.9920 - loss: 0.0274 - val_accuracy: 0.9677 - val_loss: 0.1380
Epoch 5/5
242/242 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.9957 - loss: 0.0159 - val_accuracy: 0.9722 - val_loss: 0.1472


# model evaluation

In [22]:
test_messages = [
    "Hey, are we meeting today?",
    "WIN cash now!!! Click the link",
    "Your OTP is 348921",
    "You have won $1000!",
    "Congratulations! You have won a free lottery ticket",
    "Don't forget to submit the assignment",
    "URGENT! You won a cash prize. Call immediately!",
    "My friend won a prize yesterday"
]

test_messages_tf = tf.constant(test_messages, dtype=tf.string)
test_messages_tf = embedding_layer(text_vec(test_messages_tf))
preds = cnn.predict(test_messages_tf)

for msg, p in zip(test_messages, preds):
    label = "Spam" if np.argmax(p)==1 else "Ham"
    print(f"{label} → {msg} ({p[0]:.2f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
Ham → Hey, are we meeting today? (1.00)
Spam → WIN cash now!!! Click the link (0.21)
Ham → Your OTP is 348921 (0.97)
Spam → You have won $1000! (0.20)
Spam → Congratulations! You have won a free lottery ticket (0.00)
Ham → Don't forget to submit the assignment (1.00)
Spam → URGENT! You won a cash prize. Call immediately! (0.00)
Spam → My friend won a prize yesterday (0.02)


In [23]:
print(classification_report(y_test.argmax(axis=1), cnn.predict(X_test).argmax(axis=1)))
print(f1_score(y_test.argmax(axis=1), cnn.predict(X_test).argmax(axis=1)))

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       966
           1       0.94      0.85      0.89       149

    accuracy                           0.97      1115
   macro avg       0.96      0.92      0.94      1115
weighted avg       0.97      0.97      0.97      1115

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
0.8904593639575972
